In [17]:
import scanpy as sc
import scvi
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import anndata as ad
import tqdm
import time

In [18]:
rng = np.random.default_rng(5834)

In [19]:
metrics = [
    'training_time',
    'elbo_train',
    'reconstruction_loss_train',
    'kl_local_train',
    'elbo_validation',
    'reconstruction_loss_validation',
    'kl_local_validation',
    'mean_1_lb',
    'mean_1_ub',
    'mean_2_lb',
    'mean_2_ub',
    'mean_1_point',
    'mean_2_point',
    'vars_1_lb',
    'vars_1_ub',
    'vars_2_lb',
    'vars_2_ub',
    'vars_1_point',
    'vars_2_point',
    'prsn_lb',
    'prsn_ub',
    'prsn_point',
    'sprm_lb',
    'sprm_ub',
    'sprm_point'
]

In [20]:
def model_statistics_bootstrap(model, samples=100, confidence=0.95):

    # settings
    alpha = 1 - confidence

    # sample
    counts = model.posterior_predictive_sample(n_samples=samples, gene_list=[0, 1]).todense()

    # compute mean statistics
    means = np.mean(counts, axis=0)
    mean_interval = np.quantile(means, [(alpha / 2), 1 - (alpha / 2)], axis=1)
    mean_mean = np.mean(means, axis=1)
    
    # compute variance statistics
    varx = np.var(counts, axis=0)
    varx_interval = np.quantile(varx, [(alpha / 2), 1 - (alpha / 2)], axis=1)
    varx_mean = np.mean(varx, axis=1)

    # compute correlation statistics
    prsn = np.empty((samples))
    sprm = np.empty((samples))
    for s in range(samples):
        prsn[s] = scipy.stats.pearsonr(counts[:, 0, s], counts[:, 1, s]).statistic
        sprm[s] = scipy.stats.spearmanr(counts[:, 0, s], counts[:, 1, s]).statistic
    prsn_interval = np.quantile(prsn, [(alpha / 2), 1 - (alpha / 2)], axis=0)
    sprm_interval = np.quantile(sprm, [(alpha / 2), 1 - (alpha / 2)], axis=0)
    prsn_mean = np.mean(prsn)
    sprm_mean = np.mean(sprm)

    # collect data
    data_dict = {
        'mean_interval': mean_interval,
        'mean_mean': mean_mean,
        'vars_interval': varx_interval,
        'vars_mean': varx_mean,
        'prsn_interval': prsn_interval,
        'prsn_mean': prsn_mean,
        'sprm_interval': sprm_interval,
        'sprm_mean': sprm_mean
    }

    return data_dict

In [21]:
def train_scVI_model(counts, model_kwargs={}, train_kwargs={}):
    '''
    Train scVI model on a sample from data distribution with given params
    and record metrics

    counts: training data
    model_kwargs: keyword arguments for model
    train_kwargs: keyword arguments for training
    '''

    # by default compute validation loss
    if not ('check_val_every_n_epoch' in train_kwargs.keys()):
        train_kwargs['check_val_every_n_epoch'] = 1

    # setup anndata
    counts_sparse = scipy.sparse.csr_matrix(counts)
    adata = ad.AnnData(counts)
    adata.layers["counts"] = counts_sparse
    scvi.model.SCVI.setup_anndata(adata, layer="counts")

    # create model
    model = scvi.model.SCVI(adata, **model_kwargs)

    # train model
    s = time.time()
    model.train(**train_kwargs)
    t = time.time() - s

    # compute model statistics
    stats = model_statistics_bootstrap(model)

    # collect results
    result_dict = {
        'training_time': t,
        'elbo_train': float(model.history['elbo_train'].iloc[-1, 0]),
        'reconstruction_loss_train': float(model.history['reconstruction_loss_train'].iloc[-1, 0]),
        'kl_local_train': float(model.history['kl_local_train'].iloc[-1, 0]),
        'elbo_validation': float(model.history['elbo_validation'].iloc[-1, 0]),
        'reconstruction_loss_validation': float(model.history['reconstruction_loss_validation'].iloc[-1, 0]),
        'kl_local_validation': float(model.history['kl_local_validation'].iloc[-1, 0]),
        'mean_1_lb': float(stats['mean_interval'][0, 0]),
        'mean_1_ub': float(stats['mean_interval'][1, 0]),
        'mean_2_lb': float(stats['mean_interval'][0, 1]),
        'mean_2_ub': float(stats['mean_interval'][1, 1]),
        'mean_1_point': float(stats['mean_mean'][0]),
        'mean_2_point': float(stats['mean_mean'][1]),
        'vars_1_lb': float(stats['vars_interval'][0, 0]),
        'vars_1_ub': float(stats['vars_interval'][1, 0]),
        'vars_2_lb': float(stats['vars_interval'][0, 1]),
        'vars_2_ub': float(stats['vars_interval'][1, 1]),
        'vars_1_point': float(stats['vars_mean'][0]),
        'vars_2_point': float(stats['vars_mean'][1]),
        'prsn_lb': float(stats['prsn_interval'][0]),
        'prsn_ub': float(stats['prsn_interval'][1]),
        'prsn_point': float(stats['prsn_mean']),
        'sprm_lb': float(stats['sprm_interval'][0]),
        'sprm_ub': float(stats['sprm_interval'][1]),
        'sprm_point': float(stats['sprm_mean'])
    }
    
    return result_dict

In [22]:
def scVI_analysis(data_name, model_name, model_kwargs={}, train_kwargs={}):

    # load data
    counts = np.load(f"./data/{data_name}.npy")

    # size
    repeats = counts.shape[2]

    # result dataframe
    result_df = pd.DataFrame(
        columns=metrics
    )

    # loop
    for r in tqdm.tqdm(range(repeats)):
        
        # run
        result = train_scVI_model(counts[:, :, r], model_kwargs=model_kwargs, train_kwargs=train_kwargs)

        # store results
        r_df = pd.DataFrame(result, index=[r])
        result_df = pd.concat([result_df, r_df])

    result_df.to_csv(f"./data/{data_name}-{model_name}.csv")

In [23]:
# data name
data_name = "indep-poi"

# model name
model_name = "Poi-16-2-1"

# model settings
model_kwargs = {
    'gene_likelihood': 'poisson',
    'n_hidden': 16,
    'n_latent': 2,
    'n_layers': 1,
    'use_observed_lib_size': True
}

# training settings
train_kwargs={
    'max_epochs': 100,
    #'early_stopping': True
}

scVI_analysis(data_name, model_name, model_kwargs=model_kwargs, train_kwargs=train_kwargs)

  0%|          | 0/30 [00:00<?, ?it/s]GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\trainer\connecto

Training:   0%|          | 0/100 [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


Exception raised during training. <class 'NameError'> 1


  0%|          | 0/30 [00:08<?, ?it/s]


SystemExit: 1

c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# data name
data_name = "indep-poi"

# model name
model_name = "NB-16-2-1"

# model settings
model_kwargs = {
    'gene_likelihood': 'nb',
    'n_hidden': 16,
    'n_latent': 2,
    'n_layers': 1,
    'use_observed_lib_size': True
}

# training settings
train_kwargs={
    'max_epochs': 100,
    #'early_stopping': True
}

scVI_analysis(data_name, model_name, model_kwargs=model_kwargs, train_kwargs=train_kwargs)

  0%|          | 0/10 [00:00<?, ?it/s]GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\trainer\connecto

Training:   0%|          | 0/100 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.
 10%|█         | 1/10 [01:03<09:32, 63.65s/it]GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
c:\Users\wjh20\Documents\Normalization\scVI_

Training:   0%|          | 0/100 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.
 20%|██        | 2/10 [02:11<08:49, 66.14s/it]GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
c:\Users\wjh20\Documents\Normalization\scVI_

...

`Trainer.fit` stopped: `max_epochs=100` reached.
 80%|████████  | 8/10 [09:24<02:30, 75.15s/it]GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
c:\Users\wjh20\Documents\Normalization\scVI_

Training:   0%|          | 0/100 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.
 90%|█████████ | 9/10 [10:31<01:12, 72.71s/it]GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
c:\Users\wjh20\Documents\Normalization\scVI_

Training:   0%|          | 0/100 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.
100%|██████████| 10/10 [11:36<00:00, 69.62s/it]


In [ ]:
# load data
counts = np.load("./data/real-data.npy")

# size
repeats = counts.shape[2]

# result dataframe
result_df = pd.DataFrame(
    columns=metrics
)

# loop
for r in tqdm.tqdm(range(repeats)):
    
    # run
    result = train_scVI_model(counts[:, :, r], model_kwargs=model_kwargs, train_kwargs=train_kwargs)

    # store results
    r_df = pd.DataFrame(result, index=[r])
    result_df = pd.concat([result_df, r_df])

result_df.to_csv(f"./data/real-data-NB-16-2-1.csv")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dat

Training:   0%|          | 0/100 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\train

Training:   0%|          | 0/100 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\train

...

`Trainer.fit` stopped: `max_epochs=100` reached.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\train

Training:   0%|          | 0/100 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\train

Training:   0%|          | 0/100 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.
100%|██████████| 10/10 [10:40<00:00, 64.09s/it]


# Appendix

- for training data with more than 2 genes, set `gene_list = [0, 1]` in `posterior_predictive_samples()` to only sample for the single gene pair

In [13]:
import json

In [14]:
config = {
    'data_name': 'indep-poi',
    'model_name': 'test',
    'gene_likelihood': 'poisson',
    'n_hidden': 16,
    'n_latent': 2,
    'n_layers': 1,
    'use_observed_lib_size': True,
    'max_epochs': 100,
    'early_stopping': False
}

In [15]:
with open("config.json", "w") as file:
    json.dump(config, file)

In [17]:
repeat = 0

%run -i single-repeat-run --repeat {repeat}

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
c:\Users\wjh20\Documents\Normalization\scVI_venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dat

Training:   0%|          | 0/100 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.


In [19]:
with open("config.json") as file:
    config = json.load(file)

In [21]:
pd.read_csv(f"./data/{config['data_name']}-{config['model_name']}-{repeat}.csv", index_col=0)

,training_time,elbo_train,reconstruction_loss_train,kl_local_train,elbo_validation,reconstruction_loss_validation,kl_local_validation,mean_1_lb,mean_1_ub,mean_2_lb,...,vars_2_lb,vars_2_ub,vars_1_point,vars_2_point,prsn_lb,prsn_ub,prsn_point,sprm_lb,sprm_ub,sprm_point
0,53.652349,4.672923,3.94515,0.727773,4.616287,3.941693,0.674594,7.392728,7.50684,7.481713,...,13.454087,14.298391,13.968729,13.874521,0.051744,0.081662,0.067955,0.056271,0.088454,0.071545
